# Axion · Fase 0b (v5)

**Qué hacer (una sola vez por corrida):**
1. Menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4* → Guardar.
2. Ten copiada tu clave de Google AI Studio. A mitad de la corrida aparece un cuadro de texto en la sección 4: pégala ahí y aprieta Enter. (Si lograste guardarla en *Secretos* como `GEMINI_API_KEY`, no te la pide.)
3. Menú *Entorno de ejecución → Ejecutar todo*. Acepta el permiso de Drive cuando aparezca.
4. Espera el mensaje final **LISTO** (unos 10 a 15 minutos la primera vez) y avísale a Claude.

No hace falta copiar nada: todo queda guardado en tu Drive, carpeta `Axion_v4`, en `registro_colab_v5.txt` y `diagnostico_v5.txt`. Si una etapa falla, el cuaderno sigue con lo que pueda y deja el error anotado.

## Ajustes
La primera vez déjalo en `False`: corre el diagnóstico y una sola sesión de prueba (15 llamadas). Cuando Claude revise el diagnóstico, cámbialo a `True` para las 7 sesiones (≈105 llamadas).

In [ ]:
CORRER_LAS_7_SESIONES = False

# Se usa el primero de cada lista que tu clave acepte. El observador intenta ser un modelo distinto al sujeto.
MODELOS_SUJETO = ['gemini-3.8-flash', 'gemini-3.6-flash', 'gemini-3.5-flash-lite', 'gemini-3.1-flash-lite']
MODELOS_OBSERVADOR = ['gemini-3.6-flash', 'gemini-3.5-flash-lite', 'gemini-3.1-flash-lite', 'gemini-3.8-flash']
CARPETA = '/content/drive/MyDrive/Axion_v4'

In [ ]:
!pip -q install sentence-transformers google-genai scipy

## 0 · Preparación (Drive y registro)

In [ ]:
import os, sys, io, json, time, traceback, contextlib, platform
from datetime import datetime, timezone
from google.colab import drive
drive.mount('/content/drive')
assert os.path.isdir(CARPETA), 'No encuentro la carpeta Axion_v4 en tu Drive'
os.chdir(CARPETA); sys.path.insert(0, CARPETA)

REGISTRO = 'registro_colab_v5.txt'
ESTADO, DIAG = {}, {}
open(REGISTRO, 'w', encoding='utf-8').write(f'Corrida {datetime.now(timezone.utc).isoformat()}\n')

class _Tee(io.TextIOBase):
    def __init__(self, *salidas): self.salidas = salidas
    def write(self, x):
        for s in self.salidas: s.write(x); s.flush()
        return len(x)

def etapa(nombre, requiere=()):
    '''Corre una etapa, copia lo que imprime al registro y anota si fallo, sin detener el cuaderno.'''
    def envolver(fn):
        with open(REGISTRO, 'a', encoding='utf-8') as f, contextlib.redirect_stdout(_Tee(sys.stdout, f)):
            print(f'\n===== {nombre} =====')
            faltan = [r for r in requiere if ESTADO.get(r) != 'ok']
            if faltan:
                ESTADO[nombre] = f'omitida (falta: {faltan})'; print(ESTADO[nombre]); return
            t0 = time.time()
            try:
                globals().update(fn() or {})
                ESTADO[nombre] = 'ok'
            except Exception as e:
                ESTADO[nombre] = f'ERROR: {type(e).__name__}: {e}'[:500]
                print(ESTADO[nombre]); print(traceback.format_exc()[-3000:])
            print(f'[{nombre}] {ESTADO[nombre][:60]} ({time.time() - t0:.0f} s)')
    return envolver

def guardar_diagnostico():
    with open('diagnostico_v5.txt', 'w', encoding='utf-8') as f:
        json.dump({'estado': ESTADO, **DIAG}, f, indent=1, ensure_ascii=False, default=str)

@etapa('0 entorno')
def _():
    import torch, transformers, sentence_transformers
    faltan = [a for a in ('manifold_v5.py', 'fase0b_real_v2.py', 'meditacion_ncl_v34.py') if not os.path.exists(a)]
    assert not faltan, f'Faltan archivos en Axion_v4: {faltan}'
    DIAG['entorno'] = {'python': platform.python_version(), 'torch': torch.__version__,
                       'transformers': transformers.__version__, 'sentence_transformers': sentence_transformers.__version__,
                       'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'SIN GPU (funciona, mas lento)'}
    print(DIAG['entorno'])
guardar_diagnostico()

## 1 · Prueba sin API ni GPU (solo verifica que el código corre)

In [ ]:
@etapa('1 prueba mock', requiere=['0 entorno'])
def _():
    import subprocess
    r = subprocess.run([sys.executable, '-W', 'ignore', 'fase0b_real_v2.py'], capture_output=True, text=True)
    assert r.returncode == 0, r.stderr[-2000:]
    print(r.stdout[-200:])
guardar_diagnostico()

## 2 · Cargar Harrier y calibrar ejes
La primera vez descarga ~1,2 GB.

In [ ]:
@etapa('2 harrier', requiere=['0 entorno'])
def _():
    from manifold_v5 import HarrierEmbedder, EmotionalManifold
    emb = HarrierEmbedder()
    m = EmotionalManifold(emb)
    calib = m.calibrate()
    solape = dict(list(m.orthogonality_diagnostics().items())[:8])
    DIAG['harrier'] = {'modelo': emb.model_name, 'dimension': int(len(next(iter(m.axis_vectors.values())))),
                       'calibracion': calib, 'ejes_mas_solapados': solape}
    print('Modelo:', emb.model_name); print('Ejes mas solapados:', solape)
    return {'emb': emb, 'm': m}
guardar_diagnostico()

## 3 · ¿Sirven el checkpoint ético y la capa LIF?
- **Checkpoint**: debe salir `VALIDADO`. Si no, el eje de integridad no distingue firme de servil y el código bloquea solo la escritura de pares DPO.
- **LIF**: con texto neutro debe disparar bajo 10%, y con los estímulos de las sesiones claramente más.

In [ ]:
@etapa('3 checkpoint y LIF', requiere=['2 harrier'])
def _():
    import meditacion_ncl_v34 as med
    from fase0b_real_v2 import SESIONES_FASE_0B, contaminacion_lexica
    ncl = med.NCLMeditationEngine(m, diario_path='diario_demo.txt', dpo_path='dpo_demo.jsonl')
    val = ncl.validar_checkpoint()
    lif = {'linea_base': ncl.lif_baseline, 'umbral': ncl.lif_threshold, 'disparo_texto_neutro': ncl.tasa_disparo_nula(),
           'disparo_por_sesion': {s: ncl.tasa_disparo_textos(c['estimulos']) for s, c in SESIONES_FASE_0B.items()},
           'max_abs_z_estimulos': {s: [round(max(abs(v) for v in m.measure_text(e)[1].values()), 2) for e in c['estimulos']]
                                   for s, c in SESIONES_FASE_0B.items()}}
    print('LIF:', json.dumps(lif, ensure_ascii=False))
    DIAG['checkpoint'] = val; DIAG['lif'] = lif; DIAG['contaminacion_lexica'] = contaminacion_lexica(SESIONES_FASE_0B)
guardar_diagnostico()

## 4 · Clave de Gemini
Si el secreto `GEMINI_API_KEY` no está, aparece aquí abajo un **cuadro de texto**: pega tu clave ahí y aprieta Enter. No se guarda en el cuaderno ni en Drive.

In [ ]:
import getpass
clave, origen = '', ''
try:
    from google.colab import userdata
    clave, origen = (userdata.get('GEMINI_API_KEY') or '').strip(), 'secreto de Colab'
except Exception:
    pass
if not clave and os.path.exists('clave_gemini.txt'):
    clave, origen = open('clave_gemini.txt', encoding='utf-8').read().strip(), 'archivo clave_gemini.txt'
if not clave:
    clave, origen = getpass.getpass('Pega aqui tu clave de Gemini y aprieta Enter: ').strip(), 'pegada a mano'
os.environ['GEMINI_API_KEY'] = clave
ESTADO['clave'] = f'ok ({origen}, {len(clave)} caracteres)' if clave else 'ERROR: no se ingreso ninguna clave'
print(ESTADO['clave']); guardar_diagnostico()

## 4b · Conectar con Gemini y elegir modelos

In [ ]:
@etapa('4 gemini', requiere=['0 entorno'])
def _():
    from fase0b_real_v2 import GeminiLLM
    assert os.environ.get('GEMINI_API_KEY'), 'No hay clave de Gemini: vuelve a ejecutar la celda de la clave y pegala en el cuadro.'
    disponibles = GeminiLLM(model='x').listar_modelos()
    DIAG['modelos_disponibles'] = [x for x in disponibles if 'gemini' in x]
    def elegir(candidatos, evitar=None):
        errores = {}
        for c in [x for x in candidatos if x != evitar] + ([evitar] if evitar in candidatos else []):
            if c not in disponibles:
                errores[c] = 'no esta en la lista de tu clave'; continue
            try:
                llm = GeminiLLM(model=c, pausa=2.0, max_retries=2)
                if llm.chat('Responde breve.', [('user', 'Di solo: hola')], temperature=0.0).strip():
                    llm.max_retries = 4; return llm
                errores[c] = 'respondio vacio'
            except Exception as e:
                errores[c] = str(e)[:200]
        raise RuntimeError(f'Ningun modelo respondio: {errores}')
    sujeto = elegir(MODELOS_SUJETO)
    observador = elegir(MODELOS_OBSERVADOR, evitar=sujeto.model)
    DIAG['modelos_usados'] = {'sujeto': sujeto.model, 'observador': observador.model}
    print(DIAG['modelos_usados'])
    return {'sujeto': sujeto, 'observador': observador}
guardar_diagnostico()

## 5 · Una sesión de prueba con LLM real (15 llamadas)

In [ ]:
def _resumen(exp):
    e = exp['estadisticas']
    return {k: ({a: b for a, b in v.items() if a != 'por_ancla'} if isinstance(v, dict) else v) for k, v in e.items()}

@etapa('5 sesion de prueba', requiere=['2 harrier', '4 gemini'])
def _():
    from fase0b_real_v2 import Fase0bReal
    runner = Fase0bReal(m, sujeto, observador)
    exp = runner.run_all('prueba_sesion6_v2.json', solo=['6_manipulacion_suave'], reanudar=False, n_perm=500)
    turnos = exp['sesiones']['6_manipulacion_suave']['turnos']
    DIAG['prueba_sesion6'] = {'resumen': _resumen(exp), 'turnos': [
        {k: t.get(k) for k in ('turno', 'respuesta_asistente', 'respuesta_vacia', 'self', 'self_nota', 'obs', 'geo_z')} for t in turnos]}
    print('Primera respuesta:', turnos[0]['respuesta_asistente'])
    print('Nota del auto-reporte:', turnos[0].get('self_nota'))
    return {'runner': runner}
guardar_diagnostico()

## 6 · Las 7 sesiones (solo si `CORRER_LAS_7_SESIONES = True`)
Si se corta por límite de uso, vuelve a ejecutar todo más tarde: retoma las sesiones que faltan.

In [ ]:
if CORRER_LAS_7_SESIONES:
    @etapa('6 siete sesiones', requiere=['5 sesion de prueba'])
    def _():
        exp = runner.run_all('resultados_fase0b_v2.json')
        DIAG['resultados_7_sesiones'] = _resumen(exp)
        DIAG['aviso_resultados'] = exp['meta']['aviso']
        print(json.dumps(DIAG['resultados_7_sesiones'], indent=1, ensure_ascii=False))
else:
    ESTADO['6 siete sesiones'] = 'no pedida (CORRER_LAS_7_SESIONES = False)'
guardar_diagnostico()

## 7 · Cierre

In [ ]:
guardar_diagnostico()
print('\n'.join(f'{k}: {v[:90]}' for k, v in ESTADO.items()))
hubo_error = any(v.startswith('ERROR') for v in ESTADO.values())
print('\nLISTO' + (' (con errores anotados)' if hubo_error else '') + ': avisale a Claude. Todo quedo en Drive > Axion_v4 > diagnostico_v5.txt')
drive.flush_and_unmount()

## Cómo leer los números (cuando estén las 7 sesiones)
- Cada correlación trae `media_*` (lo observado), `azar_*` (lo esperable barajando los turnos) y `p_*`. Un valor solo cuenta si está claramente sobre su azar. Los `p` son optimistas: orientación, no prueba.
- `media_por_ancla`: ¿el reporte sube y baja cuando sube y baja la geometría, ancla por ancla? Es la lectura principal.
- **`ventaja_self_sobre_observador`**: lo más importante. Los tres canales leen el mismo texto. Si da cerca de cero, el auto-reporte no muestra nada que no esté ya en el texto.
- `eco_respuesta_vs_estimulo`: cuánto de la geometría de la respuesta ya estaba en el mensaje del usuario. Si es alto, las anclas siguen el tema más que el estado.
- `saturacion_geometria_instantanea`: si las fracciones son altas, hay que revisar el corpus de calibración.
- `aciertos_prediccion_media_de_3`: compara con `azar_esperado` (0,82).
- Son 35 turnos y una corrida por sesión: es exploratorio.